In [1]:
!pip install langchain_google_genai



In [12]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")

file_path = r"S:\BitBuddy\ACE26\AgenticAI\fastapi-course\1. Introduction to APIs\Module 1 notes.pdf"
loader = PyPDFLoader(file_path)

In [13]:
docs = loader.load()
docs

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-07-12T08:45:03+00:00', 'source': 'S:\\BitBuddy\\ACE26\\AgenticAI\\fastapi-course\\1. Introduction to APIs\\Module 1 notes.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content="An API (Application Programming Interface) is essentially a set of rules and protocols that allows different \nsoftware applications to communicate with each other\n•\nIt can be thought of as a bridge that connects different systems or services, enabling them to share data \nand functionalities without directly interacting with each other's underlying code\n•\nEx: Weather app•\n1. Definition\nSunday, January 19, 2025 7:51 AM\n   1. Intro to APIs Page 4"),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-07-12T08:45:03+00:00', 'source': 'S:\\BitBuddy\\ACE26\\AgenticAI\\fastapi-course\\1. Introduction to APIs\\Module 1 notes.pdf', 'total_pages': 17,

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)

document=text_splitter.split_documents(docs)


In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",google_api_key=os.getenv("GOOGLE_API_KEY"))

In [17]:

from langchain_chroma import Chroma
fapi=Chroma.from_documents(documents=document,embedding=embeddings)

In [19]:
query="What is API?"
ans=fapi.similarity_search(query)
ans[0].page_content

"An API (Application Programming Interface) is essentially a set of rules and protocols that allows different \nsoftware applications to communicate with each other\n•\nIt can be thought of as a bridge that connects different systems or services, enabling them to share data \nand functionalities without directly interacting with each other's underlying code\n•\nEx: Weather app•\n1. Definition\nSunday, January 19, 2025 7:51 AM\n   1. Intro to APIs Page 4"

In [42]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model="models/gemini-2.5-flash",api_key=os.getenv("GOOGLE_API_KEY"))


In [26]:
!pip install langchain-classic   

In [43]:

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    <s>[INST] You are a helpful assistant that can answer questions about the provided documents. [/INST]"
    [INST] {context} [/INST]
    [INST] {input} [/INST]
    """
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    <s>[INST] You are a helpful assistant that can answer questions about the provided documents. [/INST]"\n    [INST] {context} [/INST]\n    [INST] {input} [/INST]\n    '), additional_kwargs={})])
| ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url

In [44]:
retriever=fapi.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001D99C0202F0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    <s>[INST] You are a helpful assistant that can answer questions about the provided documents. [/INST]"\n    [INST] {cont

In [45]:
response=retrieval_chain.invoke({"input": "What is the full form of API?"})
response

{'input': 'What is the full form of API?',
 'context': [Document(id='9fb8f02c-1750-4b59-8379-011688fed39f', metadata={'page': 0, 'source': 'S:\\BitBuddy\\ACE26\\AgenticAI\\fastapi-course\\1. Introduction to APIs\\Module 1 notes.pdf', 'moddate': '2025-07-12T08:45:03+00:00', 'producer': 'iLovePDF', 'total_pages': 17, 'page_label': '1', 'creator': 'PyPDF', 'creationdate': ''}, page_content="An API (Application Programming Interface) is essentially a set of rules and protocols that allows different \nsoftware applications to communicate with each other\n•\nIt can be thought of as a bridge that connects different systems or services, enabling them to share data \nand functionalities without directly interacting with each other's underlying code\n•\nEx: Weather app•\n1. Definition\nSunday, January 19, 2025 7:51 AM\n   1. Intro to APIs Page 4"),
  Document(id='9a7e1f94-9367-49b9-b05e-d9f743b6b602', metadata={'source': 'S:\\BitBuddy\\ACE26\\AgenticAI\\fastapi-course\\1. Introduction to APIs\\M